# RAPID example_id matching to gauging station names

In [24]:
import geopandas as gpd
import pandas as pd

import plotly.graph_objects as go

import plotly.express as px

In [411]:
p = '/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/'
df_ex = gpd.read_file(p +'hierarchy_examples_filtered_subdaily_manual_updates_final.gpkg')

dfsub = pd.read_parquet(p + f'subdaily_values/BR/subdaily_timeseries.parquet')
# dfsub2 = pd.read_parquet(p + f'subdaily_values/LA/subdaily_timeseries.parquet')

In [412]:
stations = df_ex.loc[df_ex['example_ids'] == '3', 'station_key'].to_list()
display(stations)

['BR:3652880', 'BR:3652890']

In [413]:
stat1 = dfsub[dfsub['station_key'] == stations[0]]
stat2 = dfsub[dfsub['station_key'] == stations[1]]
# stat3 = dfsub[dfsub['station_key'] == stations[2]]
# stat4 = dfsub[dfsub['station_key'] == stations[3]]

# stat3 = dfsub2[dfsub2['station_key'] == stations[2]]


# RAPID Hydrograph Peak selection

In [414]:
p1 = stat1.copy()
p2 = stat2.copy()
# p3 = stat3.copy()
# p4 = stat4.copy()


tstart = '2023-02-13 00:01:00+0000'
tend   = '2023-02-22 00:00:00+0000'
p1 = p1[(p1['time'] > tstart) & (p1['time'] < tend)]
p2 = p2[(p2['time'] > tstart) & (p2['time'] < tend)]
# p3 = p3[(p3['time'] > tstart) & (p3['time'] < tend)]
# p4 = p4[(p4['time'] > tstart) & (p4['time'] < tend)]

stationList = [p1,p2]
# stationList = [p1,p2]
# stationList = [p1,p2, p3]

In [415]:
fig = go.Figure()
for p in stationList:
    fig.add_trace(go.Scatter(
        x=p["time"],
        y=p["discharge"],
        mode="markers",
        name=p['station_key'].iloc[0],
        hovertemplate=f"{p['station_key'].iloc[0]}" +"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
    ))

# fig.add_trace(go.Scatter(
#     x=p2["time"],
#     y=p2["discharge"],
#     mode="markers",
#     name=f"{p2['station_key'].iloc[0]}",
#     hovertemplate=f"{p2['station_key'].iloc[0]}"+"<br>time: %{x}<br>discharge: %{y}<extra></extra>"
# ))

fig.update_layout(
    title="Discharge over Time",
    xaxis_title="time",
    yaxis_title="discharge"
)

fig.show('browser')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


Example 03:
- Peak  tstart = '2023-02-12 00:00:00+0000' tend   = '2023-02-23 00:00:00+0000'
- Exact start 2023-02-14T18:00:00Z --> 8 hour window
- Exact end 2023-02-18T16:00:00Z --> 10 hour window

## Normalize peak files to 30 min or one hour for RAPID
- make sure all timesteps are equal

In [292]:
tstart = "2023-02-12 00:00:00+0000"
tend   = "2023-02-23 00:00:00+0000"

p1 = stat1.copy()
p1["time"] = pd.to_datetime(p1["time"], utc=True)

p1 = p1[(p1["time"] > tstart) & (p1["time"] < tend)].copy()
p1 = p1.sort_values("time")

# set time index
p1i = p1.set_index("time")

# 30-minute discharge series
p1_30min = (
    p1i["discharge"]
    .resample("30min")
    .mean()
    .interpolate("time")
    .reset_index()
)

# hourly discharge series
p1_1h = (
    p1i["discharge"]
    .resample("1h")
    .mean()
    .interpolate("time")
    .reset_index()
)

p1_30min.to_csv(p + "subdaily_values/BR/Ex_03_2023_02_30min.csv", index=False)
p1_1h.to_csv(p + "subdaily_values/BR/Ex_03_2023_02_1h.csv", index=False)


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('float64'), dtype('<U42')) -> None

# RAPID Peak plot

In [293]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import netcdf_file

exp_dir = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/sarl03_indep")
run_registry = pd.read_csv(exp_dir / "rapid_run_registry.csv")

series_by_state = {}

for row in run_registry.itertuples(index=False):
    if row.status != "ran":
        continue

    qout_path = Path(row.qout_nc)
    prep_manifest_path = Path(row.rapid_prep_dir) / "rapid_prep_manifest.json"

    with netcdf_file(qout_path, "r", mmap=False) as ds:
        rivid = np.array(ds.variables["rivid"].data).copy()
        time = np.array(ds.variables["time"].data).copy()
        qout = np.array(ds.variables["Qout"].data).copy()

    riv = pd.read_csv(Path(row.rapid_prep_dir) / "riv.csv", header=None)[0].to_numpy()
    final_reach_id = int(riv[-1])
    final_idx = int(np.where(rivid == final_reach_id)[0][0])

    ts = pd.DataFrame(
        {
            "time_seconds": time,
            "q_cms": qout[:, final_idx],
        }
    )
    ts["state_id"] = row.state_id
    ts["final_reach_id"] = final_reach_id

    series_by_state[row.state_id] = ts

# example: one state
print(series_by_state["S001_unit_5"].head())

# combine all states into one long table
all_outlet_ts = pd.concat(series_by_state.values(), ignore_index=True)
print(all_outlet_ts.head())


   time_seconds  q_cms     state_id  final_reach_id
0             0    0.0  S001_unit_5             796
1          1800    0.0  S001_unit_5             796
2          3600    0.0  S001_unit_5             796
3          5400    0.0  S001_unit_5             796
4          7200    0.0  S001_unit_5             796
   time_seconds  q_cms   state_id  final_reach_id
0             0    0.0  S000_base             807
1          1800    0.0  S000_base             807
2          3600    0.0  S000_base             807
3          5400    0.0  S000_base             807
4          7200    0.0  S000_base             807


In [294]:
tstart = pd.Timestamp(tstart)

fig = go.Figure()

for k, u in series_by_state.items():
    start = 0
    end = 450

    t_seconds = np.asarray(u["time_seconds"][start:end], dtype=np.float64).copy()
    q = np.asarray(u["q_cms"][start:end], dtype=np.float64).copy()
    state_name = u["state_id"].iloc[0]

    t_hours = t_seconds / 3600.0
    t_datetime = tstart + pd.to_timedelta(t_hours, unit="h")

    fig.add_trace(
        go.Scatter(
            x=t_datetime,
            y=q,
            mode="markers",
            name=state_name,
            customdata=np.column_stack([t_hours]),
            hovertemplate=(
                f"{state_name}"
                + "<br>time: %{x|%Y-%m-%d %H:%M}"
                + "<br>hours since start: %{customdata[0]:.1f}"
                + "<br>discharge: %{y:.2f}"
                + "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Discharge over Time",
    xaxis_title="time",
    yaxis_title="discharge",
)

fig.show(renderer="browser")


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


# RAPID Hydrograph Analysis

Use this notebook after `RAPID/run_rapid_experiment.py` has finished. It reads the per-state outlet hydrographs and hydrograph metrics written under each state's `rapid/run/` folder.

In [45]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [46]:
experiment_dir = Path("/Users/6256481/Code/river-hierarchy/network_variants/output/sarl03_indep")
run_registry = pd.read_csv(experiment_dir / "rapid_run_registry.csv")
run_registry.head()

,state_id,rapid_prep_dir,rapid_run_dir,qout_nc,status,hydrograph_status,outlet_hydrograph_csv,hydrograph_metrics_csv,hydrograph_metrics_json,event_start_source,...,outlet_excess_volume_m3,outlet_reach_count,outlet_reach_ids,hydrograph_duration_seconds,metric_config,fall_time_seconds,fall_time_50_seconds,fall_time_10_seconds,e_folding_time_seconds,rise_rate_cms_per_hour
0,S000_base,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,2.536446e+08,1,[807],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",15919.863862,10865.470836,14989.529239,12326.779611,10.272216
1,S001_unit_5,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,2.561641e+08,1,[796],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",16011.401452,11196.489790,15128.120693,12617.240607,10.362339
2,S002_unit_7,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,2.596719e+08,1,[798],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",15544.705415,11094.531576,14744.726161,12405.272673,10.464934
3,S003_unit_20,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,2.548129e+08,1,[797],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",16912.721221,11967.368089,16019.545571,13406.549233,10.336321
4,S004_unit_8,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,2.562650e+08,1,[795],948600.0,"{""event_start_time"": ""2023-02-14T18:00:00Z"", ""...",15948.166175,11143.558088,15068.410104,12559.054722,10.362237


In [47]:
metric_cols = [
    "state_id",
    "event_start_time_utc",
    "peak_time_utc",
    "event_end_time_utc",
    "peak_discharge_cms",
    "peak_excess_cms",
    "peak_excess_to_end_baseline_cms",
    "time_to_peak_seconds",
    "event_duration_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
    "peak_attenuation_ratio",
]
run_registry[metric_cols].sort_values("peak_discharge_cms", ascending=False).head(28)

,state_id,event_start_time_utc,peak_time_utc,event_end_time_utc,peak_discharge_cms,peak_excess_cms,peak_excess_to_end_baseline_cms,time_to_peak_seconds,event_duration_seconds,fall_time_seconds,fall_time_50_seconds,fall_time_10_seconds,e_folding_time_seconds,lag_to_inflow_peak_seconds,peak_attenuation_ratio
23,S023_unit_27,2023-02-14T19:00:00+00:00,2023-02-19T04:30:00+00:00,2023-02-19T01:00:00+00:00,2601.428743,1138.033339,36.279399,379800.0,367200.0,11345.507581,7987.714057,10776.333734,9025.196805,160200.0,0.990909
13,S013_unit_10,2023-02-14T19:00:00+00:00,2023-02-19T05:30:00+00:00,2023-02-19T01:00:00+00:00,2600.587417,1119.113212,54.548526,383400.0,367200.0,13981.972251,9730.437561,13215.295029,10998.778961,163800.0,0.990589
18,S018_unit_19,2023-02-14T19:00:00+00:00,2023-02-19T06:00:00+00:00,2023-02-19T01:00:00+00:00,2600.435474,1110.529667,65.046711,385200.0,367200.0,15446.001927,10743.484070,14598.167835,12103.212963,165600.0,0.990531
12,S012_unit_14,2023-02-14T19:00:00+00:00,2023-02-19T06:00:00+00:00,2023-02-19T01:00:00+00:00,2600.432445,1110.007305,65.726769,385200.0,367200.0,15653.303332,10914.919776,14792.735169,12292.470270,165600.0,0.990530
21,S021_unit_9,2023-02-14T19:00:00+00:00,2023-02-19T06:30:00+00:00,2023-02-19T01:00:00+00:00,2600.380291,1103.597750,74.538545,387000.0,367200.0,16493.873795,11333.971599,15544.102732,12839.720853,167400.0,0.990510
27,S027_unit_17,2023-02-14T19:00:00+00:00,2023-02-19T05:00:00+00:00,2023-02-19T01:00:00+00:00,2600.319657,1127.271146,45.252421,381600.0,367200.0,12820.837392,9023.661864,12129.411579,10117.406582,162000.0,0.990487
6,S006_unit_15,2023-02-14T19:00:00+00:00,2023-02-19T06:30:00+00:00,2023-02-19T01:00:00+00:00,2600.101801,1103.366862,74.382116,387000.0,367200.0,16514.660049,11349.112824,15564.571397,12856.516135,167400.0,0.990404
2,S002_unit_7,2023-02-14T19:00:00+00:00,2023-02-19T05:30:00+00:00,2023-02-19T01:00:00+00:00,2600.073595,1114.515431,59.188134,383400.0,367200.0,15544.705415,11094.531576,14744.726161,12405.272673,163800.0,0.990393
26,S026_unit_25,2023-02-14T19:00:00+00:00,2023-02-19T05:30:00+00:00,2023-02-19T01:00:00+00:00,2600.009643,1115.330939,58.133304,383400.0,367200.0,15220.946466,10828.193972,14440.071392,12099.715139,163800.0,0.990369
1,S001_unit_5,2023-02-14T19:00:00+00:00,2023-02-19T06:00:00+00:00,2023-02-19T01:00:00+00:00,2599.983894,1108.770297,66.574586,385200.0,367200.0,16011.401452,11196.489790,15128.120693,12617.240607,165600.0,0.990359


In [48]:
def load_state_hydrograph(state_id: str) -> pd.DataFrame:
    row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
    hydro = pd.read_csv(row["outlet_hydrograph_csv"])
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True)
    return hydro

state_id = run_registry.loc[run_registry["status"].eq("ran"), "state_id"].iloc[0]
hydro = load_state_hydrograph(state_id)
hydro.head()

,time_seconds,time_utc,q_outlet_cms,q_inflow_cms
0,0.0,2023-02-12 00:00:00+00:00,0.0,1631.680
1,1800.0,2023-02-12 00:30:00+00:00,0.0,1628.365
2,3600.0,2023-02-12 01:00:00+00:00,0.0,1628.365
3,5400.0,2023-02-12 01:30:00+00:00,0.0,1625.050
4,7200.0,2023-02-12 02:00:00+00:00,0.0,1625.050


In [49]:
row = run_registry.loc[run_registry["state_id"].eq(state_id)].iloc[0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_inflow_cms"], mode="lines", name="Inflow"))
fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_outlet_cms"], mode="lines", name="Outlet"))
fig.add_vline(x=pd.to_datetime(row["event_start_time_utc"], utc=True), line_dash="dash", line_color="gray")
fig.add_vline(x=pd.to_datetime(row["peak_time_utc"], utc=True), line_dash="dot", line_color="red")
fig.update_layout(title=f"{state_id} outlet hydrograph", xaxis_title="time", yaxis_title="discharge (cms)")
fig.show()

In [50]:
summary = run_registry.copy()
for column in [
    "time_to_peak_seconds",
    "fall_time_seconds",
    "fall_time_50_seconds",
    "fall_time_10_seconds",
    "e_folding_time_seconds",
    "lag_to_inflow_peak_seconds",
]:
    summary[column.replace("_seconds", "_hours")] = pd.to_numeric(summary[column], errors="coerce") / 3600.0

px.scatter(
    summary,
    x="time_to_peak_hours",
    y="peak_discharge_cms",
    hover_name="state_id",
    color="peak_attenuation_ratio",
    title="Peak magnitude vs. time to peak",
)

## Merge RAPID outputs with base network metrics

For `independent-units`, each child state corresponds to one collapsed base unit. The merge below combines:

- `rapid_run_registry.csv`
- `transition_registry.csv`
- base `collapse_ranking.csv`

so you can compare outlet hydrograph behavior against the original unit metrics such as `equivalent_length`, `elongation`, and `topologic_complexity_score`.

In [93]:
transition_registry = pd.read_csv(experiment_dir / "transition_registry.csv")
base_collapse_ranking = pd.read_csv(experiment_dir / "states" / "S000_base" / "hierarchy" / "collapse_ranking.csv")

def parse_single_unit_id(value):
    if pd.isna(value):
        return pd.NA
    parts = [part.strip() for part in str(value).split(",") if part.strip()]
    if len(parts) != 1:
        return pd.NA
    return int(parts[0])

transition_registry["unit_id"] = transition_registry["selected_unit_ids"].apply(parse_single_unit_id).astype("Int64")
base_collapse_ranking["unit_id"] = pd.to_numeric(base_collapse_ranking["unit_id"], errors="coerce").astype("Int64")

analysis = summary.merge(
    transition_registry[["child_state_id", "unit_id", "collapsed_selection_label", "selected_collapse_order", "selected_collapse_priority_score"]],
    left_on="state_id",
    right_on="child_state_id",
    how="left",
)
analysis = analysis.merge(
    base_collapse_ranking,
    on="unit_id",
    how="left",
    suffixes=("", "_network"),
)

analysis["unit_id"] = analysis["unit_id"].astype("Int64")
analysis.head()

,state_id,rapid_prep_dir,rapid_run_dir,qout_nc,status,hydrograph_status,outlet_hydrograph_csv,hydrograph_metrics_csv,hydrograph_metrics_json,event_start_source,...,unit_node_ids,unit_node_count,unit_topodynamic_class,path_disparity_width_rank_component,elongation_rank_component,equivalent_length_rank_component,topologic_complexity_score_rank_component,collapse_priority_score,collapse_order_global,collapse_order_in_bubble
0,S000_base,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S001_unit_5,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,"9,11",2.0,dominant_simple_split,0.961538,0.961538,0.923077,0.826923,0.918269,1.0,1.0
2,S002_unit_7,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,"8,12",2.0,dominant_simple_split,0.846154,1.000000,1.000000,0.826923,0.918269,2.0,1.0
3,S003_unit_20,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,"38,39",2.0,dominant_simple_split,0.884615,0.923077,0.615385,0.826923,0.812500,3.0,1.0
4,S004_unit_8,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,ran,computed,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,manual_input_min_window,...,"13,15",2.0,dominant_simple_split,1.000000,0.807692,0.500000,0.826923,0.783654,4.0,1.0


In [94]:
comparison_cols = [
    "state_id",
    "unit_id",
    "collapsed_selection_label",
    "peak_discharge_cms",
    "peak_excess_cms",
    "time_to_peak_hours",
    "peak_attenuation_ratio",
    "equivalent_length",
    "elongation",
    "effective_n_paths_width",
    "path_disparity_width",
    "topologic_complexity_score",
    "collapse_priority_score",
    "collapse_order_global",
]
analysis[comparison_cols].sort_values("peak_discharge_cms", ascending=False)

,state_id,unit_id,collapsed_selection_label,peak_discharge_cms,peak_excess_cms,time_to_peak_hours,peak_attenuation_ratio,equivalent_length,elongation,effective_n_paths_width,path_disparity_width,topologic_complexity_score,collapse_priority_score,collapse_order_global
23,S023_unit_27,27,collapsed_unit_27,2601.428743,1138.033339,105.5,0.990909,2108.976781,5.048737,1.981164,1.009508,1.098612,0.331731,23.0
13,S013_unit_10,10,collapsed_unit_10,2600.587417,1119.113212,106.5,0.990589,1808.637678,4.089860,1.821393,1.098061,1.098612,0.475962,13.0
18,S018_unit_19,19,collapsed_unit_19,2600.435474,1110.529667,107.0,0.990531,1540.928059,1.915527,2.711047,1.106584,3.178054,0.432692,18.0
12,S012_unit_14,14,collapsed_unit_14,2600.432445,1110.007305,107.0,0.990530,1717.578789,4.836537,1.799208,1.111600,1.098612,0.495192,12.0
21,S021_unit_9,9,collapsed_unit_9,2600.380291,1103.597750,107.5,0.990510,1258.304901,3.755072,1.975014,1.012651,1.791759,0.370192,21.0
27,S027_unit_17,17,collapsed_unit_17,2600.319657,1127.271146,106.0,0.990487,3427.371582,6.833349,2.963196,1.012420,3.178054,0.105769,27.0
6,S006_unit_15,15,collapsed_unit_15,2600.101801,1103.366862,107.5,0.990404,1244.884461,2.501807,1.710167,1.169476,1.098612,0.649038,6.0
2,S002_unit_7,7,collapsed_unit_7,2600.073595,1114.515431,106.5,0.990393,240.123073,0.528292,1.791349,1.116477,1.098612,0.918269,2.0
26,S026_unit_25,25,collapsed_unit_25,2600.009643,1115.330939,106.5,0.990369,2417.974287,9.357555,1.998381,1.000810,2.197225,0.115385,26.0
1,S001_unit_5,5,collapsed_unit_5,2599.983894,1108.770297,107.0,0.990359,448.350226,1.130936,1.598921,1.250844,1.098612,0.918269,1.0


In [95]:
network_metric_columns = [
    "equivalent_length",
    "elongation",
    "effective_n_paths_width",
    "path_disparity_width",
    "topologic_complexity_score",
    "collapse_priority_score",
]

plot_df = analysis.loc[analysis["unit_id"].notna()].copy()
plot_df["unit_id"] = plot_df["unit_id"].astype(int)
plot_df["state_label"] = plot_df["state_id"] + " | unit " + plot_df["unit_id"].astype(str)

network_vs_peak = plot_df.melt(
    id_vars=["state_id", "state_label", "unit_id", "peak_discharge_cms", "peak_attenuation_ratio", "time_to_peak_hours"],
    value_vars=network_metric_columns,
    var_name="network_metric",
    value_name="network_value",
).dropna(subset=["network_value"])

px.scatter(
    network_vs_peak,
    x="network_value",
    y="peak_discharge_cms",
    facet_col="network_metric",
    facet_col_wrap=3,
    hover_name="state_label",
    color="peak_attenuation_ratio",
    title="Peak discharge vs. selected network metrics",
)

In [96]:
corr_columns = [
    "peak_discharge_cms",
    "peak_excess_cms",
    "time_to_peak_hours",
    "fall_time_50_hours",
    "peak_attenuation_ratio",
    "equivalent_length",
    "elongation",
    "effective_n_paths_width",
    "path_disparity_width",
    "topologic_complexity_score",
    "collapse_priority_score",
]

corr_df = analysis.loc[analysis["unit_id"].notna(), corr_columns].apply(pd.to_numeric, errors="coerce")
corr = corr_df.corr(numeric_only=True)

px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation between RAPID hydrograph metrics and network metrics",
)

## Expanded network predictors and bubble-membership analysis

This second analysis block keeps the original heatmap above, but broadens the network side with additional width and topology metrics and swaps the RAPID side to focus on:

- `time_to_peak_hours`
- `fall_time_10_hours`
- `peak_attenuation_ratio`
- `lag_to_inflow_peak_hours`
- `outlet_excess_volume_m3`
- `rise_rate_cms_per_hour`

It also adds a separate look at `time_to_peak` vs bubble membership.

In [99]:
analysis = analysis.copy()
analysis["lag_to_inflow_peak_hours"] = pd.to_numeric(analysis["lag_to_inflow_peak_seconds"], errors="coerce") / 3600.0
analysis["bubble_membership"] = analysis["in_compound_bubble"].map({True: "bubble_member", False: "standalone"}).fillna("unknown")

rapid_metric_columns_expanded = [
    "time_to_peak_hours",
    "fall_time_10_hours",
]

network_metric_candidates_expanded = [
    "bifurcation",
    "confluence",
    "equivalent_width",
    "equivalent_length",
    "n_paths",
    "n_valid_paths",
    "elongation",
    "path_length_min",
    "path_length_max",
    "path_length_mean",
    # "path_length_range",
    "path_length_range_norm",
    "path_length_cv",
    "path_width_eq_min",
    # "path_width_eq_max",
    "path_width_eq_mean",
    "path_width_range",
    "path_width_range_norm",
    "largest_path_width_fraction",
    "dominant_width_fraction",
    "width_entropy",
    "width_evenness",
    "effective_n_paths_width",
    "path_disparity_width",
    "width_ratio_2",
    "smaller_width_fraction_2",
    "dominant_width_fraction_2",
    "length_ratio_2",
    "internal_bifurcation_count",
    # "internal_confluence_count",
    "total_bifurcation_count",
    # "total_confluence_count",
    "internal_branch_node_count",
    "branching_density_by_length",
    "path_redundancy",
    "compound_indicator",
    # "topologic_complexity_score",
    "dynamic_proxy_entropy",
    "effective_n_paths_dyn_width",
    "dominant_dyn_fraction_width",
    "dynamic_proxy_complexity_score",
    "unit_node_count",
    "depth_from_root",
    "collapse_level",
    "n_children",
    "n_descendants",
    # "path_disparity_width_rank_component",
    # "elongation_rank_component",
    # "equivalent_length_rank_component",
    # "topologic_complexity_score_rank_component",
    "collapse_priority_score",
]

available_network_metrics_expanded = [
    column for column in network_metric_candidates_expanded if column in analysis.columns
]

available_network_metrics_expanded

['bifurcation',
 'confluence',
 'equivalent_width',
 'equivalent_length',
 'n_paths',
 'n_valid_paths',
 'elongation',
 'path_length_min',
 'path_length_max',
 'path_length_mean',
 'path_length_range_norm',
 'path_length_cv',
 'path_width_eq_min',
 'path_width_eq_mean',
 'path_width_range',
 'path_width_range_norm',
 'largest_path_width_fraction',
 'dominant_width_fraction',
 'width_entropy',
 'width_evenness',
 'effective_n_paths_width',
 'path_disparity_width',
 'width_ratio_2',
 'smaller_width_fraction_2',
 'dominant_width_fraction_2',
 'length_ratio_2',
 'internal_bifurcation_count',
 'total_bifurcation_count',
 'internal_branch_node_count',
 'branching_density_by_length',
 'path_redundancy',
 'compound_indicator',
 'dynamic_proxy_entropy',
 'effective_n_paths_dyn_width',
 'dominant_dyn_fraction_width',
 'dynamic_proxy_complexity_score',
 'unit_node_count',
 'depth_from_root',
 'collapse_level',
 'n_children',
 'n_descendants',
 'collapse_priority_score']

In [100]:
corr_columns_expanded = rapid_metric_columns_expanded + available_network_metrics_expanded
corr_df_expanded = analysis.loc[analysis["unit_id"].notna(), corr_columns_expanded].apply(pd.to_numeric, errors="coerce")
corr_expanded = corr_df_expanded.corr(numeric_only=True)
cross_corr = corr_expanded.loc[rapid_metric_columns_expanded, available_network_metrics_expanded]

fig = px.imshow(
    cross_corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Cross-domain correlations: time to peak and fall time vs. network metrics",
)
fig.update_xaxes(title="Network metrics")
fig.update_yaxes(title="RAPID metrics")
fig

In [ ]:
- equivalent length
- elongation
- 

In [101]:
cross_domain_rows = []
for rapid_metric in rapid_metric_columns_expanded:
    for network_metric in available_network_metrics_expanded:
        corr_value = cross_corr.loc[rapid_metric, network_metric]
        cross_domain_rows.append(
            {
                "rapid_metric": rapid_metric,
                "network_metric": network_metric,
                "correlation": corr_value,
                "abs_correlation": abs(corr_value),
            }
        )

cross_domain_corr = pd.DataFrame(cross_domain_rows).sort_values("abs_correlation", ascending=False)
cross_domain_corr.head(20)

,rapid_metric,network_metric,correlation,abs_correlation
49,fall_time_10_hours,path_length_min,-0.641245,0.641245
45,fall_time_10_hours,equivalent_length,-0.634115,0.634115
51,fall_time_10_hours,path_length_mean,-0.628794,0.628794
7,time_to_peak_hours,path_length_min,-0.610631,0.610631
50,fall_time_10_hours,path_length_max,-0.608166,0.608166
3,time_to_peak_hours,equivalent_length,-0.607216,0.607216
9,time_to_peak_hours,path_length_mean,-0.603179,0.603179
8,time_to_peak_hours,path_length_max,-0.587904,0.587904
12,time_to_peak_hours,path_width_eq_min,-0.482334,0.482334
48,fall_time_10_hours,elongation,-0.472643,0.472643


In [102]:
px.bar(
    cross_domain_corr.head(15),
    x="abs_correlation",
    y="network_metric",
    color="rapid_metric",
    orientation="h",
    hover_data=["correlation"],
    title="Strongest RAPID vs. network correlations",
)

In [103]:
bubble_df = analysis.loc[analysis["unit_id"].notna()].copy()
bubble_df["unit_id"] = bubble_df["unit_id"].astype(int)
bubble_df["state_label"] = bubble_df["state_id"] + " | unit " + bubble_df["unit_id"].astype(str)

px.box(
    bubble_df,
    x="bubble_membership",
    y="time_to_peak_hours",
    color="bubble_membership",
    points="all",
    hover_name="state_label",
    title="Time to peak by bubble membership",
)

In [104]:
px.box(
    bubble_df,
    x="bubble_membership",
    y="fall_time_10_hours",
    color="bubble_membership",
    points="all",
    hover_name="state_label",
    title="Fall time 10 by bubble membership",
)